### 01 - Attack (Feature Extraction + MIA Classifier)

Extracts 5 attack features from shadow and target models, then trains a logistic regression attack classifier to predict membership.

For each shadow model *i*: scores its own train subset (member=1) and its unique test fold (member=0). These are aggregated into the shadow feature table for attack training. The target model is scored identically for evaluation.

##### Inputs
- `outputs/models/{target,shadow}_*.h5`
- `outputs/models/shadow_{x1,x2,y}_train_{i}.npy`
- `data/external/clinicalbert/*.npy`

##### Outputs
- `outputs/results/mia_features_{shadow,target}.csv`
- `outputs/results/mia_attack_metrics.csv`
- `outputs/figures/mia_roc_phase2.png`

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.metrics import roc_auc_score, roc_curve
from tensorflow.keras.models import load_model

from pprl_attack import (
    EMB_DIR, MODEL_DIR, RESULT_DIR, FIG_DIR, N_SHADOWS,
    score_pairs, build_feature_frame, ensure_dir,
)

ensure_dir(RESULT_DIR)
ensure_dir(FIG_DIR)
from pprl_attack.run_logger import log_run

In [ ]:
# ====================================
# Load Models
# ====================================
target_encoder = load_model(MODEL_DIR / "target_encoder.h5", compile=False)
target_clf = load_model(MODEL_DIR / "target_clf.h5", compile=False)
print("Target model loaded.")

shadow_encoders = []
shadow_clfs = []
for i in range(N_SHADOWS):
    enc = load_model(MODEL_DIR / f"shadow_encoder_{i}.h5", compile=False)
    clf = load_model(MODEL_DIR / f"shadow_clf_{i}.h5", compile=False)
    shadow_encoders.append(enc)
    shadow_clfs.append(clf)
print(f"Loaded {N_SHADOWS} shadow models.")

In [ ]:
# ====================================
# Load Embeddings
# ====================================
x1_target_train = np.load(EMB_DIR / "x1_target_train.npy")
x2_target_train = np.load(EMB_DIR / "x2_target_train.npy")
y_target_train  = np.load(EMB_DIR / "y_target_train.npy")
x1_target_test  = np.load(EMB_DIR / "x1_target_test.npy")
x2_target_test  = np.load(EMB_DIR / "x2_target_test.npy")
y_target_test   = np.load(EMB_DIR / "y_target_test.npy")

x1_shadow_test = np.load(EMB_DIR / "x1_shadow_test.npy")
x2_shadow_test = np.load(EMB_DIR / "x2_shadow_test.npy")
y_shadow_test  = np.load(EMB_DIR / "y_shadow_test.npy")
test_fold_indices = np.load(MODEL_DIR / "test_fold_indices.npy", allow_pickle=True)

print(f"Target train: {x1_target_train.shape[0]} pairs")
print(f"Target test:  {x1_target_test.shape[0]} pairs")
print(f"Shadow test:  {x1_shadow_test.shape[0]} pairs")

In [ ]:
# ====================================
# Extract Shadow Features
# ====================================
shadow_frames = []

for i in range(N_SHADOWS):
    x1_tr = np.load(MODEL_DIR / f"shadow_x1_train_{i}.npy")
    x2_tr = np.load(MODEL_DIR / f"shadow_x2_train_{i}.npy")
    y_tr  = np.load(MODEL_DIR / f"shadow_y_train_{i}.npy")

    probs_tr = score_pairs(shadow_encoders[i], shadow_clfs[i], x1_tr, x2_tr)
    te_idx = test_fold_indices[i]
    y_te_i = y_shadow_test[te_idx]
    probs_te = score_pairs(
        shadow_encoders[i], shadow_clfs[i],
        x1_shadow_test[te_idx], x2_shadow_test[te_idx],
    )

    shadow_frames.append(
        build_feature_frame(probs_tr, y_tr, member_label=1, source_name=f"shadow_{i}_train")
    )
    shadow_frames.append(
        build_feature_frame(probs_te, y_te_i, member_label=0, source_name=f"shadow_test_{i}")
    )

shadow_features = pd.concat(shadow_frames, ignore_index=True)
print(f"Shadow features: {len(shadow_features)} rows")
print(f"  members:    {(shadow_features['member']==1).sum()}")
print(f"  non-members:{(shadow_features['member']==0).sum()}")
print("\nMean loss by membership (shadow):")
print(shadow_features.groupby('member')['loss'].agg(['mean', 'std']))

In [ ]:
# ====================================
# Extract Target Features
# ====================================
probs_target_train = score_pairs(target_encoder, target_clf, x1_target_train, x2_target_train)
probs_target_test  = score_pairs(target_encoder, target_clf, x1_target_test, x2_target_test)

target_features = pd.concat([
    build_feature_frame(probs_target_train, y_target_train, member_label=1, source_name="target_train"),
    build_feature_frame(probs_target_test, y_target_test, member_label=0, source_name="target_test"),
], ignore_index=True)

print(f"Target features: {len(target_features)} rows")
print(f"  members:    {(target_features['member']==1).sum()}")
print(f"  non-members:{(target_features['member']==0).sum()}")
print("\nMean loss by membership (target):")
print(target_features.groupby('member')['loss'].agg(['mean', 'std']))

# Save CSVs
shadow_features.to_csv(RESULT_DIR / "mia_features_shadow.csv", index=False)
target_features.to_csv(RESULT_DIR / "mia_features_target.csv", index=False)
print(f"\nSaved: {RESULT_DIR / 'mia_features_shadow.csv'}")
print(f"Saved: {RESULT_DIR / 'mia_features_target.csv'}")

In [ ]:
# ====================================
# Train Attack Classifier
# ====================================
features = ["prob", "loss", "correctness_confidence", "entropy", "prob_correct"]

X_shadow = shadow_features[features].values
y_shadow = shadow_features["member"].values
X_target = target_features[features].values
y_target = target_features["member"].values

X_strain, X_sval, y_strain, y_sval = train_test_split(
    X_shadow, y_shadow, test_size=0.2, stratify=y_shadow, random_state=42,
)
print(f"Shadow train: {len(y_strain)}  Shadow val: {len(y_sval)}")

lr = LogisticRegression(class_weight="balanced", max_iter=2000)
lr_gs = GridSearchCV(lr, {"C": [0.01, 0.1, 1, 10, 100]}, cv=5, scoring="roc_auc", n_jobs=-1)
lr_gs.fit(X_strain, y_strain)
lr_best = lr_gs.best_estimator_

lr_val_prob = lr_best.predict_proba(X_sval)[:, 1]
lr_val_auc = roc_auc_score(y_sval, lr_val_prob)
lr_targ_prob = lr_best.predict_proba(X_target)[:, 1]
lr_targ_auc = roc_auc_score(y_target, lr_targ_prob)
coefs = lr_best.coef_[0]
print(f"LR  best_C={lr_gs.best_params_['C']}  val_AUC={lr_val_auc:.4f}  targ_AUC={lr_targ_auc:.4f}")
print(f"    coefs: " + "  ".join(f"{f}={c:+.4f}" for f, c in zip(features, coefs)))

# Bootstrap confidence interval
n_bootstrap = 1000
rng = np.random.RandomState(42)
n_target = len(y_target)
bootstrap_aucs = np.zeros(n_bootstrap)
for b in range(n_bootstrap):
    idx = rng.choice(n_target, size=n_target, replace=True)
    bootstrap_aucs[b] = roc_auc_score(y_target[idx], lr_targ_prob[idx])
ci_low = np.percentile(bootstrap_aucs, 2.5)
ci_high = np.percentile(bootstrap_aucs, 97.5)
final_auc = roc_auc_score(y_target, lr_targ_prob)
print(f"LR target AUC: {final_auc:.4f}  (95% CI: [{ci_low:.4f}, {ci_high:.4f}])")

In [ ]:
# ====================================
# Evaluate: TPR at FPR thresholds
# ====================================
fpr, tpr, _ = roc_curve(y_target, lr_targ_prob)
tpr_1 = np.interp(0.01, fpr, tpr)
tpr_10 = np.interp(0.10, fpr, tpr)
print(f"TPR@FPR=1%:  {tpr_1:.4f}")
print(f"TPR@FPR=10%: {tpr_10:.4f}")

metrics = {
    "attack_auc": final_auc,
    "auc_ci_low": ci_low,
    "auc_ci_high": ci_high,
    "tpr_at_fpr_1pct": tpr_1,
    "tpr_at_fpr_10pct": tpr_10,
}
pd.DataFrame([metrics]).to_csv(RESULT_DIR / "mia_attack_metrics.csv", index=False)
print(f"Saved: {RESULT_DIR / 'mia_attack_metrics.csv'}")

In [ ]:
# ====================================
# ROC Curve
# ====================================
plt.figure(figsize=(6, 6))
fpr_lr, tpr_lr, _ = roc_curve(y_target, lr_targ_prob)
plt.plot(fpr_lr, tpr_lr, lw=2, label=f"LR (AUC={final_auc:.4f})")
plt.plot([0, 1], [0, 1], "k--", alpha=0.4, label="random")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("MIA Attack ROC Curve")
plt.legend(loc="lower right")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(FIG_DIR / "mia_roc_phase2.png", dpi=120)
plt.show()

# --- metrics + run log ---
from sklearn.metrics import accuracy_score, confusion_matrix

y_pred = lr_best.predict(X_target)
acc = accuracy_score(y_target, y_pred)
tn, fp, fn, tp = confusion_matrix(y_target, y_pred).ravel()
majority_baseline = max(y_target.mean(), 1 - y_target.mean())

coef_str = ", ".join(f"{f}={c:+.4f}" for f, c in zip(features, coefs))

# Within-model MIA (shadow-free baseline)
X_wm = target_features[features].values
y_wm = target_features["member"].values
X_wm_tr, X_wm_te, y_wm_tr, y_wm_te = train_test_split(
    X_wm, y_wm, test_size=0.3, random_state=42, stratify=y_wm,
)
wm_clf = LogisticRegression(class_weight="balanced", max_iter=2000).fit(X_wm_tr, y_wm_tr)
within_auc = roc_auc_score(y_wm_te, wm_clf.predict_proba(X_wm_te)[:, 1])

log_run("01_attack",
    shadow_rows=len(shadow_features),
    target_rows=len(target_features),
    n_features=len(features),
    feature_names=", ".join(features),
    best_C=lr_gs.best_params_["C"],
    cv_score=round(float(lr_val_auc), 4),
    attack_auc=round(float(final_auc), 4),
    auc_ci_95=f"[{ci_low:.4f}, {ci_high:.4f}]",
    bootstrap_n=n_bootstrap,
    tpr_at_fpr_1pct=round(float(tpr_1), 4),
    tpr_at_fpr_10pct=round(float(tpr_10), 4),
    attack_accuracy=round(float(acc), 4),
    majority_class_baseline=round(float(majority_baseline), 4),
    within_model_auc=round(float(within_auc), 4),
    confusion_matrix=f"tn={tn} fp={fp} fn={fn} tp={tp}",
    lr_coefficients=coef_str,
)